# Januário Leads — Próxima Colab V1: triagem de lotes para Geoapify (SIMULAÇÃO)

**Preparada em 16/11/2026.** Esta Colab é a continuação segura das importações Overture, **não reinicia** a Fase 9 V6.1 nem consulta APIs externas.

### Situação de referência (consulta somente leitura à homologação)
- **321.179** empresas na tabela `public.overture_places`.
- **38** tentativas registradas em `public.overture_geoapify_enrichment`, todas com `status=no_match`.
- **50** créditos contabilizados como **reservados** em `public.overture_geoapify_free_credit_usage`, não equivalentes a custo faturado.
- Esses números são um retrato de 16/11/2026, não se atualizam automaticamente nesta Colab.

### Segurança obrigatória
A **Fase 0 ainda está pendente**: revogar a antiga chave `service_role` de homologação exposta e verificar backup/restauração de produção. Por isso este notebook **não pede senhas/chaves**, **não acessa Supabase**, **não consulta Geoapify, Google Places nem OpenAI** e **não grava em bancos**. Executar todas as células em `DEMONSTRACAO` apenas prepara 25 empresas fictícias; modo `CSV` prepara candidatos a partir de arquivos exportados por canal seguro.

A próxima execução *real* de enriquecimento deverá ser implementada/autorizada **separadamente** após Fase 0, confirmação dos termos do provedor, cota e reserva atômica. **Não interprete 25 candidatos como 25 créditos gratuitos garantidos.**

### Saídas
`lote_geoapify_revisao.csv` (candidatos, não enriquecimento), `cobertura_amostra.csv` (resumo por UF/cidade) e `manifesto_lote.json` (contagens e bloqueios). **Não publicar o CSV empresarial em repositório público.**

In [ ]:
# 1 — Configuração. "Executar tudo" não envia solicitações externas.
MODO_ENTRADA = "DEMONSTRACAO"  # "DEMONSTRACAO" ou "CSV"
ARQUIVO_OVERTURE = "overture_places_candidatos.csv"
ARQUIVO_TENTATIVAS = "overture_geoapify_enrichment.csv"

MAX_LINHAS_CSV = 5000
MAX_CANDIDATOS = 25
MAX_POR_CIDADE = 3
MAX_POR_UF = 8
PERMITIR_REQUISICOES_EXTERNAS = False
PERMITIR_ESCRITA_SUPABASE = False

assert MODO_ENTRADA in ("DEMONSTRACAO", "CSV")
assert MAX_CANDIDATOS <= 25 and MAX_POR_CIDADE <= 3 and MAX_POR_UF <= 8
if PERMITIR_REQUISICOES_EXTERNAS or PERMITIR_ESCRITA_SUPABASE:
    raise RuntimeError("Bloqueio Fase 0: esta Colab não realiza consultas ou escritas.")
print("MODO SEGURO:", MODO_ENTRADA, "| APIs externas: 0 | Escritas DB: 0")

In [ ]:
# 2 — Funções puras de triagem. Biblioteca padrão Python; nenhuma rede.
import collections
import csv
import datetime as dt
import json
import re
import unicodedata
from pathlib import Path

COLUNAS = ("id","name","category","city","state","address","latitude",
           "longitude","phone","email","website","instagram","operating_status")
COLUNAS_TENTATIVAS = ("overture_id","status")
PESOS_AUSENCIA = {"phone":5,"email":4,"website":3,"instagram":2,"address":1}
UFS = set("AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO".split())

def texto(valor):
    if valor is None: return ""
    return str(valor).strip()

def chave(valor):
    x = unicodedata.normalize("NFD", texto(valor).lower())
    x = "".join(c for c in x if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9]+", " ", x).strip()

def campo_ausente(registro, campo):
    return not texto(registro.get(campo))

def pontuacao(registro):
    return sum(peso for campo,peso in PESOS_AUSENCIA.items()
               if campo_ausente(registro,campo))

def coordenadas_validas(registro):
    try:
        lat=float(texto(registro.get("latitude")))
        lon=float(texto(registro.get("longitude")))
        return -34 <= lat <= 6 and -75 <= lon <= -28
    except (TypeError,ValueError):
        return False

def elegivel(registro):
    return bool(texto(registro.get("id")) and
                texto(registro.get("name")) and
                texto(registro.get("category")) and
                texto(registro.get("city")) and
                texto(registro.get("state")).upper() in UFS and
                coordenadas_validas(registro) and
                texto(registro.get("operating_status")).lower() != "permanently_closed"
                and pontuacao(registro) > 0)

def ler_csv_limitado(arquivo, obrigatorias):
    caminho=Path(arquivo)
    if not caminho.is_file():
        raise FileNotFoundError("CSV não encontrado: "+str(caminho))
    if caminho.stat().st_size > 25_000_000:
        raise RuntimeError("CSV acima de 25 MB; exporte amostras menores, não toda a base.")
    with caminho.open(encoding="utf-8-sig",newline="") as fp:
        leitor=csv.DictReader(fp)
        if not set(obrigatorias).issubset(set(leitor.fieldnames or [])):
            raise ValueError("Colunas CSV obrigatórias: "+", ".join(obrigatorias))
        dados=[]
        for linha in leitor:
            if len(dados) >= MAX_LINHAS_CSV:
                raise RuntimeError("CSV ultrapassou o limite de 5.000 linhas; reduza a amostra.")
            dados.append(linha)
        return dados

def salvar_csv(arquivo,registros,colunas):
    # Proteção contra fórmulas em ferramentas de planilha: campos são dados não confiáveis.
    def excel_seguro(v):
        x=texto(v)
        if re.fullmatch(r"-?[0-9]+(?:\\.[0-9]+)?", x): return x
        return "'"+x if x and x[0] in ("=","+","-","@","\t","\r","\n") else x
    with Path(arquivo).open("w",encoding="utf-8-sig",newline="") as fp:
        escritor=csv.DictWriter(fp,fieldnames=colunas,extrasaction="ignore")
        escritor.writeheader()
        for row in registros:
            escritor.writerow({col:excel_seguro(row.get(col,"")) for col in colunas})

def amostra_ficticia():
    cidades=(("MG","Lavras"),("MG","Varginha"),("MG","Pouso Alegre"),
            ("SP","Campinas"),("SP","Santos"),("SP","Sorocaba"),
            ("RJ","Niterói"),("RJ","Petrópolis"),("RJ","Cabo Frio"),
            ("PR","Curitiba"),("PR","Maringá"),("PR","Londrina"))
    dados=[]
    for i in range(144):
        uf,municipio=cidades[i%len(cidades)]
        dados.append({
            "id":f"FAKE_{i:04d}","name":f"Empresa fictícia {i:04d}",
            "category":"barber" if i%2 else "gym",
            "city":municipio,"state":uf,"address":"" if i%5==0 else f"Rua Fictícia {i}",
            "latitude":str(-21.2+(i%8)*.01),"longitude":"-44.96",
            "phone":"" if i%4 else "35999990000",
            "email":"" if i%3 else "naoexiste@example.invalid",
            "website":"" if i%2 else "https://example.invalid",
            "instagram":"","operating_status":"",
        })
    tentativas=[{"overture_id":"FAKE_0000","status":"no_match"},
               {"overture_id":"FAKE_0001","status":"no_match"},
               {"overture_id":"FAKE_0002","status":"matched"}]
    return dados,tentativas

print("Funções carregadas. Nenhuma chamada de rede realizada.")

In [ ]:
# 3 — Fonte dos dados. Em CSV, use apenas exports autorizados da homologação.
# Não cole credenciais neste notebook; não use a chave service_role exposta.
if MODO_ENTRADA == "DEMONSTRACAO":
    empresas,tentativas = amostra_ficticia()
else:
    if not Path(ARQUIVO_OVERTURE).exists() or not Path(ARQUIVO_TENTATIVAS).exists():
        try:
            from google.colab import files
            print("Envie OS DOIS CSVs: Overture e tentativas já realizadas.")
            files.upload()
        except ImportError:
            print("Fora do Colab: copie os dois arquivos CSV para a pasta atual.")
    # Bloqueio: não continuar se não tivermos histórico, para não repetir consultas.
    empresas = ler_csv_limitado(ARQUIVO_OVERTURE, ("id","name","city","state","category",
                                               "latitude","longitude"))
    tentativas = ler_csv_limitado(ARQUIVO_TENTATIVAS,COLUNAS_TENTATIVAS)
    if not tentativas:
        raise RuntimeError("Histórico vazio: exporte as tentativas anteriores para evitar repetições.")
print("Linhas de entrada:",len(empresas),"| tentativas históricas fornecidas:",len(tentativas))
print("Isto é uma amostra local, NÃO uma leitura em tempo real do Supabase.")

In [ ]:
# 4 — Seleção conservadora e balanceada: exclui QUALQUER tentativa anterior.
historicos={texto(t.get("overture_id")) for t in tentativas if texto(t.get("overture_id"))}
vistos=set()
descartes=collections.Counter()
candidatos=[]
for e in empresas:
    identificador=texto(e.get("id"))
    if not identificador or identificador in vistos:
        descartes["id_ausente_ou_duplicado"]+=1
        continue
    vistos.add(identificador)
    if identificador in historicos:
        descartes["tentativa_ja_registrada"]+=1
        continue
    if not elegivel(e):
        descartes["incompleto_ou_sem_contatos_pendentes"]+=1
        continue
    registro={col:texto(e.get(col)) for col in COLUNAS}
    registro["priority_score"]=pontuacao(e)
    candidatos.append(registro)

candidatos.sort(key=lambda e:(-e["priority_score"],e["state"],
                              chave(e["city"]),e["id"]))
por_uf=collections.defaultdict(list)
for e in candidatos: por_uf[e["state"]].append(e)
selecionados=[]
contador_uf=collections.Counter()
contador_cidade=collections.Counter()
while len(selecionados)<MAX_CANDIDATOS and any(por_uf.values()):
    houve_adicao=False
    for uf in sorted(por_uf):
        if len(selecionados)>=MAX_CANDIDATOS: break
        if contador_uf[uf]>=MAX_POR_UF: continue
        fila=por_uf[uf]
        while fila and contador_cidade[(uf,chave(fila[0]["city"]))]>=MAX_POR_CIDADE:
            fila.pop(0)
            descartes["limite_por_cidade"]+=1
        if not fila: continue
        e=fila.pop(0)
        selecionados.append(e)
        contador_uf[uf]+=1
        contador_cidade[(uf,chave(e["city"]))]+=1
        houve_adicao=True
    if not houve_adicao: break

assert len(selecionados)<=25
assert all(e["id"] not in historicos for e in selecionados)
assert len({e["id"] for e in selecionados})==len(selecionados)
assert all(n<=MAX_POR_UF for n in contador_uf.values())
assert all(n<=MAX_POR_CIDADE for n in contador_cidade.values())
print("Elegíveis na amostra:",len(candidatos))
print("Preparados para REVISÃO (não consulta):",len(selecionados))
print("Descartes por motivo:",dict(descartes))
print("Cotas por UF (amostra):",dict(contador_uf))

In [ ]:
# 5 — Resumo seguro, sem divulgar nomes, números ou perfis no log
agrupados=collections.defaultdict(list)
for e in empresas:
    agrupados[(texto(e.get("state")).upper(),texto(e.get("city")))].append(e)
distribuicao=collections.Counter((e["state"],e["city"]) for e in selecionados)
cobertura=[]
for (uf,municipio),registros in sorted(agrupados.items()):
    cobertura.append({
        "state":uf,"city":municipio,"rows_in_sample":len(registros),
        "phone_not_recorded":sum(campo_ausente(x,"phone") for x in registros),
        "email_not_recorded":sum(campo_ausente(x,"email") for x in registros),
        "website_not_recorded":sum(campo_ausente(x,"website") for x in registros),
        "instagram_not_recorded":sum(campo_ausente(x,"instagram") for x in registros),
        "batch_count":distribuicao[(uf,municipio)],
    })
print("Municípios na amostra:",len(cobertura))
print("Ausência de informação NÃO comprova que o contato ou site não exista.")
print("Nenhum telefone, e-mail ou nome empresarial é exibido nas saídas de texto.")

In [ ]:
# 6 — Gerar arquivos locais no Colab, sem gravar em nenhuma API ou banco
import hashlib
saida=Path("/content" if Path("/content").is_dir() else ".")/"januario_proxima_colab"
saida.mkdir(parents=True,exist_ok=True)
campos_lote=("id","name","category","city","state","address","latitude",
             "longitude","phone","email","website","instagram",
             "priority_score","source","review_status")
lote=[]
for e in selecionados:
    # Dados da fonte Overture preservados; NÃO produzimos novos contatos/Instagram.
    item=dict(e)
    item["source"]="overture"
    item["review_status"]="PENDING_MANUAL_REVIEW"
    lote.append(item)
salvar_csv(saida/"lote_geoapify_revisao.csv",lote,campos_lote)
salvar_csv(saida/"cobertura_amostra.csv",cobertura,tuple(cobertura[0].keys())
           if cobertura else ("state","city","rows_in_sample","batch_count"))
manifesto={
    "notebook":"Januario_Leads_Proxima_Colab_V1_Preparacao_Geoapify_Offline",
    "generated_at_utc":dt.datetime.now(dt.timezone.utc).isoformat(),
    "mode":MODO_ENTRADA,
    "staging_read_live":False,"supabase_writes":0,
    "geoapify_requests":0,"google_places_requests":0,"openai_requests":0,
    "sample_rows":len(empresas),"history_rows":len(tentativas),
    "eligible_rows":len(candidatos),"selected_rows":len(lote),
    "max_per_run":MAX_CANDIDATOS,
    "excluded_previous_attempts":descartes["tentativa_ja_registrada"],
    "note":"Plano de revisão; nenhum enriquecimento real, cobrança ou validação de contatos.",
}
(saida/"manifesto_lote.json").write_text(
    json.dumps(manifesto,indent=2,ensure_ascii=False),encoding="utf-8")
if MODO_ENTRADA=="DEMONSTRACAO":
    assert manifesto["geoapify_requests"]==0 and manifesto["supabase_writes"]==0
    assert manifesto["selected_rows"]>0
    assert not any(x["id"] in {"FAKE_0000","FAKE_0001","FAKE_0002"} for x in lote)
for arquivo in sorted(saida.iterdir()):
    print("Criado:",arquivo.name,"-",arquivo.stat().st_size,"bytes")
print("IMPORTANTE: NÃO compartilhar arquivos com dados de empresas publicamente.")

## Como usar com dados reais, **somente depois da Fase 0**

1. **Antes de qualquer CSV real:** revogar/rotacionar a chave `service_role` comprometida, concluir backup/restauração de produção e revisar acesso à homologação. Não utilizar a chave antiga.
2. Exportar, em um ambiente autorizado, uma **amostra de até 5.000 empresas** da tabela `overture_places` e o histórico de `overture_geoapify_enrichment`; utilize os dois scripts somente leitura: `scripts/colab_proximo_lote_geoapify_candidatos_readonly.sql` e `scripts/colab_proximo_lote_geoapify_historico_readonly.sql`. Execute cada SELECT separadamente no projeto de homologação. Não usar o banco de produção.
3. Em `MODO_ENTRADA="CSV"`, enviar **ambos** os CSVs. Executar do começo ao fim. A seleção exclui todos os IDs já tentados, inclusive `no_match`, para não consumir novamente uma consulta sem revisão.
4. Revisar o `lote_geoapify_revisao.csv` local. Ele contém **apenas dados Overture existentes**, sem qualquer dado confirmado por Geoapify.
5. O enriquecimento real, quando autorizado, será feito em outro fluxo com cota e orçamento atômico do backend. Esta Colab **não** tem botão oculto para habilitar API paga.

**Limites:** este notebook não valida telefone, e-mail, site, Instagram, nem correspondência de empresas. O arquivo de saída é uma fila de conferência; não deve ser importado automaticamente sobre `overture_places`. O armazenamento e as permissões de dados empresariais devem seguir a Fase 3.1a antes de uso comercial.